# Step 5 — Load Saved Model and Predict Emotion

This notebook loads the trained SER model and preprocessing artifacts, then runs prediction on unseen `.wav` files with confidence scores, visualizations, and batch export support.

## 1. Imports and configuration

In [1]:
# ============================================================
# Reproducibility & Device Configuration (Mac CPU)
# ============================================================

import random
import numpy as np
import torch
import os
import json
import pickle

from pathlib import Path

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# Use all CPU cores
NUM_CORES = 4

os.environ["OMP_NUM_THREADS"] = str(NUM_CORES)
os.environ["MKL_NUM_THREADS"] = str(NUM_CORES)

torch.set_num_threads(NUM_CORES)
torch.set_num_interop_threads(NUM_CORES)


DEVICE = torch.device("cpu")


print("=" * 70)
print("PyTorch Version :", torch.__version__)
print("Running Device  :", DEVICE)

print("CPU Threads     :", torch.get_num_threads())
print("Inter-op Threads:", torch.get_num_interop_threads())

print("=" * 70)

PyTorch Version : 2.12.0
Running Device  : cpu
CPU Threads     : 4
Inter-op Threads: 4


In [2]:
# ============================================================
# Paths
# ============================================================

BASE_DIR = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent.parent

MODEL_DIR = PROJECT_ROOT / "models" / "wav2vec2_ser"

OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("MODEL DIR:")
print(MODEL_DIR)

print("Exists:")
print(MODEL_DIR.exists())

MODEL DIR:
/Users/devanshbansal/Desktop/ser/models/wav2vec2_ser
Exists:
True


## 2. Load prepared assets


In [3]:
# ============================================================
# Load Prepared Assets from Notebook 4
# ============================================================
from pathlib import Path
import pickle
from datasets import load_from_disk
from transformers import AutoFeatureExtractor

BASE_DIR = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
PREP_DIR = OUTPUT_DIR / "prepared_training"

print("Loading assets from:")
print(PREP_DIR)

if not PREP_DIR.exists():
    raise FileNotFoundError(
        f"\nPrepared assets not found:\n{PREP_DIR}\n"
        "Run Notebook 4 first."
    )

# Load Hugging Face datasets
train_dataset = load_from_disk(PREP_DIR / "train_dataset")
val_dataset   = load_from_disk(PREP_DIR / "val_dataset")
test_dataset  = load_from_disk(PREP_DIR / "test_dataset")

with open(
    MODEL_DIR / "label_encoder.pkl",
    "rb"
) as f:
    label_encoder = pickle.load(f)
    
# Load Feature Extractor
feature_extractor = AutoFeatureExtractor.from_pretrained(
    MODEL_DIR
)
# Load Label Encoder


print("=" * 70)
print("Prepared Assets Loaded Successfully")
print("=" * 70)

print(f"Training Samples   : {len(train_dataset)}")
print(f"Validation Samples : {len(val_dataset)}")
print(f"Testing Samples    : {len(test_dataset)}")

print(f"\nNumber of Classes  : {len(label_encoder.classes_)}")
print(f"Classes            : {list(label_encoder.classes_)}")

print("\nDataset Features:")
print(train_dataset.features)

print("=" * 70)

/opt/homebrew/Caskroom/miniforge/base/envs/malaria_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading assets from:
/Users/devanshbansal/Desktop/ser/outputs/prepared_training
Prepared Assets Loaded Successfully
Training Samples   : 1134
Validation Samples : 162
Testing Samples    : 144

Number of Classes  : 7
Classes            : ['angry', 'calm', 'disgust', 'fearful', 'happy', 'sad', 'surprised']

Dataset Features:
{'waveform': List(Value('float32')), 'emotion': Value('string'), 'input_values': List(Value('float32')), 'attention_mask': List(Value('int32')), 'label': Value('int64')}


In [5]:
from pathlib import Path

for p in Path(PROJECT_ROOT).rglob("*label*encoder*"):
    print(p)

/Users/devanshbansal/Desktop/ser/models/wav2vec2_ser/label_encoder.pkl
/Users/devanshbansal/Desktop/ser/outputs/prepared_training/label_encoder.pkl
/Users/devanshbansal/Desktop/ser/notebooks/5_Load saved model Load scaler & label encoder Predict emotion from unseen audio Show confidence scores Batch prediction support Save predictions


## 3. Load model


In [ ]:
# ============================================================
# Load Wav2Vec2 Model
# ============================================================
from transformers import Wav2Vec2ForSequenceClassification
MODEL_DIR = PROJECT_ROOT / "models" / "wav2vec2_ser"
MODEL_NAME = "facebook/wav2vec2-base"
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_DIR
)
model.to(DEVICE)
model.eval()
id2label = {
    i: label
    for i, label in enumerate(label_encoder.classes_)
}

label2id = {
    label: i
    for i, label in enumerate(label_encoder.classes_)
}

print("=" * 70)
print("Wav2Vec2 Model Loaded Successfully")
print("=" * 70)

print(f"Pretrained Model : {MODEL_NAME}")
print(f"Number of Classes : {model.config.num_labels}")
print(f"Classifier Output Features: {model.classifier.out_features}")
print("\nLabel Mapping")
for idx, label in id2label.items():
    print(f"{idx:2d} -> {label}")

print("=" * 70)
model.config.final_dropout = 0.1

## 4.Freeze feature extractor


In [ ]:
# ============================================================
# Freeze Wav2Vec2 Feature Encoder
# ============================================================
# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("=" * 70)
print("Feature Encoder Frozen")
print("=" * 70)

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Frozen Parameters     : {frozen_params:,}")
print(f"Trainable Percentage  : {(trainable_params/total_params)*100:.2f}%")

print("=" * 70)

## 5. compute_metrics

In [ ]:
# ============================================================
# Evaluation Metrics
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    _, _, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "macro_f1": f1_macro,
        "weighted_f1": f1_weighted,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro
    }


print("=" * 70)
print("Evaluation Metrics Ready")
print("=" * 70)

print("Metrics:")
print("✓ Accuracy")
print("✓ Macro F1")
print("✓ Weighted F1")
print("✓ Macro Precision")
print("✓ Macro Recall")
print("=" * 70)

In [ ]:
from transformers import (
    AutoFeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer
)

## 6. TrainingArguments


In [ ]:
# ============================================================
# Training Configuration (Mac CPU)
# ============================================================

MODEL_SAVE_DIR = PROJECT_ROOT / "models" / "wav2vec2_ser"

training_args = TrainingArguments(
    output_dir=str(MODEL_SAVE_DIR),

    num_train_epochs=10,

    learning_rate=2e-5,
    warmup_ratio=0.1,

    weight_decay=0.01,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,

    fp16=False,

    logging_steps=10
)


print("=" * 70)
print("TrainingArguments Created")
print("=" * 70)

print(f"Epochs                 : {training_args.num_train_epochs}")
print(f"Batch Size             : {training_args.per_device_train_batch_size}")
print(f"Gradient Accumulation  : {training_args.gradient_accumulation_steps}")
print(f"Learning Rate          : {training_args.learning_rate}")
print(f"FP16 Enabled           : {training_args.fp16}")
print(f"Workers                : {training_args.dataloader_num_workers}")

print("=" * 70)

In [ ]:
# ============================================================
# Custom Data Collator
# ============================================================

from dataclasses import dataclass
from typing import Dict, List, Union

import torch

@dataclass
class DataCollatorCTCWithPadding:
    feature_extractor: AutoFeatureExtractor

    def __call__(
        self,
        features: List[Dict[str, Union[list, int]]]
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )

        batch["labels"] = torch.tensor(
            [feature["label"] for feature in features],
            dtype=torch.long
        )

        return batch


data_collator = DataCollatorCTCWithPadding(feature_extractor)

print("=" * 70)
print("Custom Data Collator Created")
print("=" * 70)

# Verify one batch
batch = data_collator([train_dataset[0], train_dataset[1]])

print("Batch Keys:")
for key, value in batch.items():
    print(f"{key:15s} {tuple(value.shape)} {value.dtype}")

print("=" * 70)

## 7. Trainer


In [ ]:
# ============================================================
# Initialize Hugging Face Trainer
# ============================================================

from transformers import EarlyStoppingCallback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5
        )
    ]
)

print("=" * 70)
print("Trainer Initialized Successfully")
print("=" * 70)

print(f"Training Samples   : {len(train_dataset)}")
print(f"Validation Samples : {len(val_dataset)}")
print(f"Training Device    : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU Name           : {torch.cuda.get_device_name(0)}")

print("=" * 70)

## 6. Prediction and visualization helpers

In [ ]:
# ============================================================
# Fine-Tune Wav2Vec2
# ============================================================

import time

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 70)
print("Starting Wav2Vec2 Fine-Tuning")
print("=" * 70)

print(f"Device           : {DEVICE}")
print(f"Training Samples : {len(train_dataset)}")
print(f"Validation Samples : {len(val_dataset)}")
print(f"Epochs           : {training_args.num_train_epochs}")
print(f"Batch Size       : {training_args.per_device_train_batch_size}")

print("=" * 70)

start_time = time.time()

train_result = trainer.train()

training_time = time.time() - start_time

print("=" * 70)
print("Training Completed Successfully")
print("=" * 70)

print(f"Total Training Time : {training_time/60:.2f} minutes")

print("\nTraining Metrics:")
for key, value in train_result.metrics.items():
    print(f"{key:25s}: {value}")

print("=" * 70)

## 10. Save Fine-Tuned Model


In [ ]:
# ============================================================
# Save Fine-Tuned Model
# ============================================================

FINAL_MODEL_DIR = PROJECT_ROOT / "models" / "wav2vec2_ser"

trainer.save_model(FINAL_MODEL_DIR)
feature_extractor.save_pretrained(FINAL_MODEL_DIR)

print("=" * 70)
print("Fine-Tuned Model Saved Successfully")
print("=" * 70)

print(f"Saved to:\n{FINAL_MODEL_DIR}")

## 11. Evaluate on Test Set

In [ ]:
# ============================================================
# Evaluate on Test Dataset
# ============================================================

test_metrics = trainer.evaluate(
    eval_dataset=test_dataset
)
print("=" * 70)
print("Test Results")
print("=" * 70)

for key, value in test_metrics.items():
    print(f"{key:25s}: {value:.4f}")

## 12. Save Evaluation Results

In [ ]:
# ============================================================
# Save Evaluation Results
# ============================================================

import json

RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_DIR / "test_metrics.json", "w") as f:
    json.dump(
        {k: float(v) for k, v in test_metrics.items()},
        f,
        indent=4
    )

print("=" * 70)
print("Evaluation Results Saved")
print("=" * 70)

## 13. training summary

In [ ]:
# ============================================================
# Training Summary
# ============================================================

print("=" * 70)
print("Fine-Tuning Summary")
print("=" * 70)

print(f"Model                 : {MODEL_NAME}")
print(f"Training Samples      : {len(train_dataset)}")
print(f"Validation Samples    : {len(val_dataset)}")
print(f"Testing Samples       : {len(test_dataset)}")

print(f"Epochs                : {training_args.num_train_epochs}")
print(f"Learning Rate         : {training_args.learning_rate}")
print(f"Batch Size            : {training_args.per_device_train_batch_size}")

print("\nFinal Test Metrics")

for k, v in test_metrics.items():
    print(f"{k:25s}: {v:.4f}")

print("=" * 70)

## 14. notebook completion

In [ ]:
# ============================================================
# Notebook 5 Completed
# ============================================================

print("=" * 70)
print("NOTEBOOK 5 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("Completed:")
print("✓ Wav2Vec2 Loaded")
print("✓ Feature Encoder Frozen")
print("✓ Fine-Tuned on RAVDESS")
print("✓ Best Model Saved")
print("✓ Test Evaluation Completed")
print("✓ Metrics Saved")

print("\nNext Notebook:")
print("Notebook 6 - Emotion Prediction & Inference")

print("=" * 70)